# Lab 3-1: Running Hive Queries

In this lab we will learn how to create Hive tables on Databricks and write HiveQL queries to analyze data in Hive tables.

We will also cover how to upload data to to Databricks's Unity Catalog, and how to use uploaded data to create managed tables.

**Folders/Files Created/Uploaded**
> - `/workspace/msbabigdata`
> - `/workspace/msbabigdata/dualcore`
> - `/workspace/msbabigdata/dualcore/customers` 
> - `/workspace/msbabigdata/dualcore/employees` 
> - `/workspace/msbabigdata/dualcore/products` 
> - `/workspace/msbabigdata/dualcore/orders` 
> - `/workspace/msbabigdata/dualcore/order_details`


**Database/Tables Created**
> - (Database) `dualcore`
> - (Managed table) `dualcore.customers` 
> - (Managed table) `dualcore.employees` 
> - (Managed table) `dualcore.products` 
> - (Managed table) `dualcore.orders`
> - (Managed table) `dualcore.order_details`


## Step 1. Upload data to Unit Catalog


> Note: our dataset files has a tab-deliminated format with no headers. If you have one of the support formats (.csv, .tsv, .tab, .json, .jsonl, .avro, .parquet, .txt, .xml, .xlsx, .xls, or .xlsm), you may also directly create table using **"Create or Modify a Table"**

1. Please use your browser to download the zipped data file from https://idsdl.csom.umn.edu/c/share/msba6330/dualcore_raw.zip
2. Unzip the files (keep in mind where you have unzipped them)
3. Go to New > Add or upload Data > Upload Files to a Volume
   - Drag and drop the unzipped folders/files
   - Destination volume: **+Create Volume** (if `msbabigdata/dualcore` already exists, you can simply select it)
      - Volume name: `dualcore`
      - Choose **External volume**
      - Select a catalog: **workspace**
      - Select a schema: **Create a new schema**  > **msbabigdata**  (if `msbabigdata` exists, you can simply select it)

![image_1770008586350.png](https://i.ibb.co/Swrd6fFj/image-1770008155086.png)
![image_1770008155086.png](https://i.ibb.co/wF5LjKPk/image-1770008586350.png)



## Step 2. Recreate the database (schema) and tables using scripts provided below.

> Due to many restrictions Databricks put on the Free Edition, we cannot directly create tables using HiveSQL commands. 
> We have to use PySpark scripts to create those tables using the uploaded data

### 2.1  Creating/Recreating the database 

In [ ]:
%sql
-- recreate the database/schema
DROP DATABASE IF EXISTS dualcore CASCADE;
CREATE DATABASE dualcore;

-- these will create a blank database in the `default`` workspace. 
-- If you run this again, you will also need to re-run the table creation script below

### 2.2 Creating/Recreating Tables

In [ ]:
# load customers data into a spark dataframe
df = spark.read.option("sep","\t").schema("cust_id string,fname string,lname string,address string,city string,state string,zipcode string").csv("/Volumes/workspace/msbabigdata/dualcore/customers/")
# save the dataframe as a Managed table
df.write.mode("overwrite").saveAsTable("workspace.dualcore.customers")

# load employees data into a spark dataframe
df = spark.read.option("sep","\t").schema("emp_id STRING, fname STRING,   lname STRING,  address STRING,  city STRING,  state STRING,  zipcode STRING,  job_title STRING,  email STRING,  active STRING,  salary INT").csv("/Volumes/workspace/msbabigdata/dualcore/employees/")

# save the dataframe as a Managed table
df.write.mode("overwrite").saveAsTable("workspace.dualcore.employees")

# load orders data into a spark dataframe
df = spark.read.option("sep","\t").schema("order_id INT,  cust_id INT,  order_date STRING").csv("/Volumes/workspace/msbabigdata/dualcore/orders/")

# save the dataframe as a Managed table
df.write.mode("overwrite").saveAsTable("workspace.dualcore.orders")

# load products data into a spark dataframe
df = spark.read.option("sep","\t").schema("prod_id INT,  brand STRING,  name STRING,  price INT,  cost INT,  shipping_wt INT").csv("/Volumes/workspace/msbabigdata/dualcore/products/")

# save the dataframe as a Managed table
df.write.mode("overwrite").saveAsTable("workspace.dualcore.products")

# load order_details data into a spark dataframe
df = spark.read.option("sep","\t").schema("order_id INT, prod_id INT").csv("/Volumes/workspace/msbabigdata/dualcore/order_details/")

# save the dataframe as a Managed table
df.write.mode("overwrite").saveAsTable("workspace.dualcore.order_details")

### 2.3 Verify the tables by selecting the first 10 rows.

In [ ]:
%sql
select* FROM dualcore.customers limit 10;


cust_id | fname | lname | address | city | state | zipcode
1000000 | Quentin | Shepard | 32092 West 10th Street | Prairie City | SD | 57649
1000001 | Brandon | Louis | 1311 North 2nd Street | Clearfield | IA | 50840
1000002 | Marilyn | Ham | 25831 North 25th Street | Concord | CA | 94522
1000003 | Mister | Tee | 11596 West 1st Street | Sterling | KS | 67579
1000004 | Nunya | Beeswax | 3044 West 21st Street | Phoenix | AZ | 85017
1000005 | Michael | Eller | 24326 West 11th Street | Germantown | IL | 62245
1000006 | Gerard | Franks | 356 Turner Street | Pioneer | CA | 95666
1000007 | Deborah | Carson | 4033 East 21st Street | Arbon | ID | 83212
1000008 | Thomas | Brush | 20643 North 12th Street | Poland | ME | 04274
1000009 | Bartman | Simpsonite | 1748 East Fairbanks Road | Alto Pass | IL | 62905

## Step 3: Query the tables

A bit context:

- Dualcore ran a contest in which customers posted videos of interesting ways to use their new tablets. A $5,000 prize will be awarded to the customer whose video received the highest rating. 
- However, the registration data was lost due to an RDBMS crash, and the only information we have is from the videos. The winning customer introduced herself only as "Bridget from Kansas City" in her video. 
- You will need to run a Hive query that identifies the winner's record in our customer database so that we can send her the $5,000 prize. 
- For your reference, you can find the description of tables and fields in the attached `datamodel.pdf`.



### 3.1 As this lab will use the `dualcore` database, set it to be the current database, then display the tables in this database. 

In [ ]:
%sql
use dualcore;

### 3.2 Describe the table `customers` to learn about its fields.

In [ ]:
%sql
describe dualcore.customers


col_name | data_type | comment
cust_id | string | None
fname | string | None
lname | string | None
address | string | None
city | string | None
state | string | None
zipcode | string | None

### 3.3 All you know about the winner is that her first name (`fname`) starts with `Bridg` (but it could be, for example, "Bridget", "Bridgette" or "Bridgitte") and she lives in `Kansas City` (her `city`). Use a Hive query to find out **who the winner of the $5,000 prize**. 

In [ ]:
%sql
select*from dualcore.customers
where fname like 'Bridge%'
and city = "Kansas City";

cust_id | fname | lname | address | city | state | zipcode
1139477 | Bridget | Burch | 1644 East 7th Street | Kansas City | KS | 66119

### 3.4. Find the top ten states with the most customers. 

In [ ]:
%sql
select state, count(*) as num
from dualcore.customers
group by state
order by num desc
limit 10;


state | num
CA | 76278
WA | 11379
AZ | 8247
IL | 7776
OR | 7719
MO | 7407
MN | 6539
IA | 6514
UT | 5829
ID | 5212

## Step 4: Create a table by uploading supported file formats.

1. Download `https://idsdl.csom.umn.edu/c/share/msba6330/titanic.csv`
2. Use **+New** > Add or Upload Data > Create or Modify a table > Create Table
3. Query the table, showing the count the passengers by sex.


In [ ]:
%sql WITH q AS (select state, count(*) as num
from dualcore.customers
group by state
order by num desc
limit 10) SELECT `state`,SUM(`num`) `column_694cae65178` FROM q GROUP BY `state`

In [ ]:
%sql
SELECT sex, COUNT(*) AS passenger_count
FROM default.titanic GROUP BY sex;

sex | passenger_count
male | 577
female | 314

## This is the end of this lab